## ACUTE MYELOID LUEKANEMIA PHENOTYPE FILE 
This script creates the phenotype file for SAIGE-GENE+. It integrates several inputs, including EUR-like exome sample quality control results, ancestry-specific PCs, and age and sex files. Only individuals who pass exome quality control are retained in the SAIGE-GENE+ phenotype file.
Case–control status is assigned by providing a list of person IDs for individuals designated as cases; all remaining individuals are assigned as controls. The script also restricts the dataset to individuals who have electronic health record data. All quantitative covariates are standardized to have a mean of 0 and a standard deviation of 1.
Because more samples passed exome quality control than ACAF quality control in this analysis, this phenotype file will also be used for the ACAF analysis, with samples failing ACAF quality control excluded at a later stage.

In [2]:
# Load packages 
from sklearn.preprocessing import StandardScaler
from functools import reduce
import pandas as pd 
import os 

In [3]:
%run /home/jupyter/workspace/rw-migration-aou-rw-9f879fc4/notebooks/Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: NOT FOUND
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/jupyter/.bashrc
C2024Q3R9 BQ_DATASET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = None
wb-silky-artichoke-2408.C2024Q3R9



In [4]:
# Workspace bucket 
my_bucket = 'gs://rw-migration-aou-rw-9f879fc4'

In [5]:
# Load quality control information and demographic information 
bucket=f'{my_bucket}/data/pcs/EUR_flashpca.pcs'
eur_pca = pd.read_csv(bucket, sep ='\t')

bucket=f'{my_bucket}/data/covariates/biosample_collection_age.tsv'
age = pd.read_csv(bucket, sep ='\t')

bucket=f'{my_bucket}/data/wgs_qc/analysis_samples_wgs_qc/exome/EUR-like_exome_qc.txt'
eur_exome = pd.read_csv(bucket, sep ='\t')

bucket=f'{my_bucket}/data/wgs_qc/ancestry_sd3_wgs_qc/EUR-like_cohort_info_qc.txt'
sex = pd.read_csv(bucket, sep ='\t')

In [6]:
# Assign biospecimen variable 
sex['biospecimen'] = sex['sample_source'].apply(lambda x: 1 if x == "WHOLE BLOOD" else 0 if x == "SALIVA" else 2)
sex['biospecimen'] = sex['biospecimen'].astype("int")

In [7]:
# Assign sex variable 
sex['sex'] = sex['dragen_sex_ploidy'].apply(lambda x: 1 if x == "XX" else 0 if x == "XY" else 2)
sex['sex'] = sex['sex'].astype("int")

In [8]:
bucket=f'gs://rw-migration-aou-rw-9f879fc4/data/phenotype/aml_person_ids.txt'
aml_person_df = pd.read_csv(bucket, sep ='\t')

In [9]:
bucket=f'gs://rw-migration-aou-rw-9f879fc4/data/phenotype/ehr_person_ids.txt'
ehr_person_df = pd.read_csv(bucket, sep ='\t')

In [10]:
# Create list of quality control information and demographic information 
dfs = [eur_exome.rename(columns={"IID":"person_id"}),
       age.rename(columns={"IID":"person_id"}).filter(regex="person_id|age"),
       sex.rename(columns={"IID":"person_id"}).filter(regex="person_id|sex|biospecimen"), 
       eur_pca.rename(columns={"IID":"person_id"}).filter(regex="person_id|PC"),
       ehr_person_df
       ]

In [11]:
# Combine list of quality control information and demographic information 
phenotype_file = reduce(
    lambda left, right: pd.merge(left, right, on="person_id", how="inner"),
    dfs
)

In [12]:
# Assign AML phenotype to individuals 
phenotype_file['aml_phenotype'] = phenotype_file['person_id'].isin(aml_person_df['person_id']).astype(int)

In [13]:
# Additional covariates 
phenotype_file["age_squared"] = phenotype_file["age"] ** 2
phenotype_file["age_sex"] = phenotype_file["age"] * phenotype_file["sex"]
phenotype_file["age_biospecimen"] = phenotype_file["age"] * phenotype_file["biospecimen"]

In [14]:
# Format phenotype file  
phenotype_file = (phenotype_file[['aml_phenotype',
            'sex', 'age', 'biospecimen',
            'age_squared', 'age_sex', 'age_biospecimen', 
            'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 
            'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 
            'person_id']]
)

In [15]:
# Identify columns that start with "PC" or "age"
cols_to_standardize = [col for col 
                       in phenotype_file.columns 
                       if col.startswith("PC") 
                       or col.startswith("age")]
# Standardize only the selected columns
scaler = StandardScaler()
phenotype_file[cols_to_standardize] = (
    scaler.fit_transform(phenotype_file[cols_to_standardize])
) 
# Print results of the standarization 
print("============================================")
print("Check variances of standardized covariates:")
print("============================================")
print(phenotype_file.filter(regex="PC|age").var())
print(phenotype_file.describe())
print("============================================")

Check variances of standardized covariates:
age                1.000006
age_squared        1.000006
age_sex            1.000006
age_biospecimen    1.000006
PC1                1.000006
PC2                1.000006
PC3                1.000006
PC4                1.000006
PC5                1.000006
PC6                1.000006
PC7                1.000006
PC8                1.000006
PC9                1.000006
PC10               1.000006
dtype: float64
       aml_phenotype            sex           age    biospecimen  \
count  169971.000000  169971.000000  1.699710e+05  169971.000000   
mean        0.001559       0.605039  4.188737e-16       0.924222   
std         0.039455       0.488844  1.000003e+00       0.264643   
min         0.000000       0.000000 -2.303439e+00       0.000000   
25%         0.000000       0.000000 -7.940227e-01       1.000000   
50%         0.000000       1.000000  1.821123e-01       1.000000   
75%         0.000000       1.000000  7.896915e-01       1.000000   
max  

In [16]:
# Final check for phenotype 
print("============================================")
print("Check the presence of aml phenotype:")
print("============================================")
print(phenotype_file["aml_phenotype"].value_counts())
print("============================================")

Check the presence of aml phenotype:
aml_phenotype
0    169706
1       265
Name: count, dtype: int64


In [17]:
phenotype_file.to_csv(
    'EUR_phenotype_file.txt',
    sep='\t',
    index=False
)

In [18]:
 !gcloud storage cp EUR_phenotype_file.txt gs://rw-migration-aou-rw-9f879fc4/data/saige_phenotype_files/aml_covariates_files/

Copying file://EUR_phenotype_file.txt to gs://rw-migration-aou-rw-9f879fc4/data/saige_phenotype_files/aml_covariates_files/EUR_phenotype_file.txt
  Completed files 1/1 | 47.0MiB/47.0MiB | 2.4MiB/s                             

Average throughput: 3.8MiB/s
